# cleaning data fraud labels

In [269]:
import pandas as pd

fraud_labels = pd.read_csv("fraud_labels.csv")
transactions = pd.read_csv("transactions.csv")
transactions

,transaction_id,account_id,merchant_id,transaction_date,transaction_type,channel,amount,currency,description
0,1,65820,19009.0,2025-05-26,Deposit,pos,448.44,ZAR,Transfer
1,2,9167,15505.0,2026-05-06,Purchase,atm,85.0,ZAR,Online purchase
2,3,86080,12053.0,2025-12-24,Payment,ATM,459.94,ZAR,Retail purchase
3,4,56039,10816.0,2025-10-05,Transfer,mobile,511.36,ZAR,Bill payment
4,5,69271,9603.0,2025-01-27,Deposit,Mobile,362.61,ZAR,Bill payment
...,...,...,...,...,...,...,...,...,...
1002995,760111,16022,3336.0,2026-06-04,Deposit,Online,920.44,ZAR,Transfer
1002996,989369,109728,9164.0,2026-05-19,Purchase,Online,935.24,ZAR,Salary
1002997,210561,42406,4358.0,2025-10-31,Withdrawal,POS,189.0,ZAR,Online purchase
1002998,698682,111100,12297.0,2025-04-21,Purchase,Online,1062.72,ZAR,Bill payment


In [270]:
fraud_labels.info()

fraud_labels.describe(include='all')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   transaction_id  1000000 non-null  int64 
 1   is_fraud        1000000 non-null  int64 
 2   fraud_reason    24260 non-null    object
dtypes: int64(2), object(1)
memory usage: 22.9+ MB


,transaction_id,is_fraud,fraud_reason
count,1000000.000000,1000000.000000,24260
unique,NaN,NaN,33
top,NaN,NaN,nan
freq,NaN,NaN,6034
mean,499999.788612,0.006389,NaN
std,288677.403098,0.079676,NaN
min,1.000000,0.000000,NaN
25%,249990.750000,0.000000,NaN
50%,500018.500000,0.000000,NaN
75%,750002.250000,0.000000,NaN


In [271]:
# Check fraud distribution
fraud_labels['is_fraud'].value_counts()

is_fraud
0    993611
1      6389
Name: count, dtype: int64

In [272]:
# Missing values
fraud_labels.isnull().sum()

transaction_id         0
is_fraud               0
fraud_reason      975740
dtype: int64

In [273]:
# Non-fraud transactions with a fraud reason
invalid_1 = fraud_labels[(fraud_labels['is_fraud'] == 0) & (fraud_labels['fraud_reason'].notna())]

# Fraud transactions without a fraud reason
invalid_2 = fraud_labels[(fraud_labels['is_fraud'] == 1) & (fraud_labels['fraud_reason'].isna())]

print(len(invalid_1))
print(len(invalid_2))

17902
31


In [274]:
#handling missing values
fraud_labels.loc[
(fraud_labels['is_fraud'] == 1) & (fraud_labels['fraud_reason'].isna()),
'fraud_reason'
] = 'Unknown'

In [275]:
#keep NaN or replace with a label
fraud_labels['fraud_reason'] = fraud_labels['fraud_reason'].fillna('No Fraud')

In [276]:
# removing duplicates
fraud_labels = fraud_labels.drop_duplicates()

fraud_labels[fraud_labels.duplicated() == True]


,transaction_id,is_fraud,fraud_reason


In [277]:
# ensuring ID's unique
fraud_labels['transaction_id'].nunique()

fraud_labels.shape[0]

999059

In [278]:
# finding duplicates
"fraud_labels.duplicated()"

'fraud_labels.duplicated()'

In [279]:
# correcting data types
fraud_labels['transaction_id'] = fraud_labels['transaction_id'].astype(float)
fraud_labels['is_fraud'] = fraud_labels['is_fraud'].astype(float)
fraud_labels.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999059 entries, 0 to 999999
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   transaction_id  999059 non-null  float64
 1   is_fraud        999059 non-null  float64
 2   fraud_reason    999059 non-null  object 
dtypes: float64(2), object(1)
memory usage: 30.5+ MB


In [280]:
# standardazing
fraud_labels['fraud_reason'] = (
fraud_labels['fraud_reason']
.str.strip()
.str.lower()
)
fraud_labels

,transaction_id,is_fraud,fraud_reason
0,1.0,0.0,no fraud
1,2.0,0.0,no fraud
2,3.0,0.0,no fraud
3,4.0,0.0,no fraud
4,5.0,0.0,no fraud
...,...,...,...
999995,999996.0,0.0,no fraud
999996,999997.0,0.0,no fraud
999997,999998.0,0.0,no fraud
999998,999999.0,0.0,no fraud


In [281]:
# Validation
print(fraud_labels.isnull().sum())
print(fraud_labels['is_fraud'].value_counts())

# Check for invalid combinations again
fraud_labels[
((fraud_labels['is_fraud'] == 0) & (fraud_labels['fraud_reason'] != 'No Fraud'))
]

transaction_id    0
is_fraud          0
fraud_reason      0
dtype: int64
is_fraud
0.0    992670
1.0      6389
Name: count, dtype: int64


,transaction_id,is_fraud,fraud_reason
0,1.0,0.0,no fraud
1,2.0,0.0,no fraud
2,3.0,0.0,no fraud
3,4.0,0.0,no fraud
4,5.0,0.0,no fraud
...,...,...,...
999995,999996.0,0.0,no fraud
999996,999997.0,0.0,no fraud
999997,999998.0,0.0,no fraud
999998,999999.0,0.0,no fraud


# Cleaning data merchants

In [282]:
import pandas as pd

merchants = pd.read_csv("merchants.csv")
merchants = pd.read_csv("merchants.csv")
merchants.info()
merchants

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   merchant_id        20000 non-null  int64 
 1   merchant_name      20000 non-null  object
 2   merchant_category  19881 non-null  object
 3   province           20000 non-null  object
 4   city               19878 non-null  object
 5   risk_rating        19869 non-null  object
dtypes: int64(1), object(5)
memory usage: 937.6+ KB


,merchant_id,merchant_name,merchant_category,province,city,risk_rating
0,1,merchant 00001,Online Services,Free State,Bethlehem,Low
1,2,Merchant 00002,Grocery,Mpumalanga,Secunda,Low
2,3,Merchant 00003,Clothing,Limpopo,Polokwane,Medium
3,4,Merchant 00004,Grocery,Mpumalanga,Mbombela,Low
4,5,Merchant 00005,Travel,Free State,Bloemfontein,Medium
...,...,...,...,...,...,...
19995,19996,Merchant 19996,Online Services,Mpumalanga,Mbombela,Low
19996,19997,Merchant 19997,Restaurant,KwaZulu-Natal,Durban,Low
19997,19998,Merchant 19998,Grocery,KwaZulu-Natal,Richards Bay,Low
19998,19999,Merchant 19999,Grocery,Free State,Welkom,Low


In [283]:
# #This helps identify spelling inconsistencies.

merchants.city = merchants.city.str.strip()
# # merchants the white and space bar space
merchants['city'] = merchants['city'].str.replace(r'\s+', ' ', regex= True)
merchants.city = merchants.city.str.title()

# merchants.city.unique()

merchants.city.unique()

array(['Bethlehem', 'Secunda', 'Polokwane', 'Mbombela', 'Bloemfontein',
       'Pietermaritzburg', 'Johannesburg', 'Gqeberha', 'Klerksdorp',
       'Stellenbosch', 'Welkom', 'Rustenburg', 'Mahikeng', 'Thohoyandou',
       'Emalahleni', 'Durban', 'Midrand', 'Richards Bay', 'East London',
       'Pretoria', 'Upington', 'Mthatha', 'George', 'Paarl', 'Soweto',
       'Kimberley', 'Tzaneen', nan, 'Kuruman', 'Cape Town'], dtype=object)

In [284]:
merchants.province= merchants.province.str.strip()
# # merchants the white and space bar space
merchants['province'] = merchants['province'].str.replace("  ", " ")
merchants.province = merchants.province.str.title()


# #Expected provinces
merchants.province.unique()

array(['Free State', 'Mpumalanga', 'Limpopo', 'Kwazulu-Natal', 'Gauteng',
       'Eastern Cape', 'North West', 'Western Cape', 'Northern Cape'],
      dtype=object)

In [ ]:
merchants.merchant_category= merchants.merchant_category.str.strip()
# # merchants the white and space bar space
merchants['merchant_category'] = merchants['merchant_category'].str.replace("  ", " ")
merchants.merchant_category = merchants.merchant_category.str.title()
merchants.merchant_category.unique()


array(['Online Services', 'Grocery', 'Clothing', 'Travel',
       'Entertainment', 'Fuel', 'Healthcare', 'Electronics', 'Other',
       'Restaurant', nan], dtype=object)

In [ ]:
merchants.risk_rating= merchants.risk_rating.str.strip()
# # merchants the white and space bar space
merchants['risk_rating'] = merchants['risk_rating'].str.replace("  ", " ")
merchants.risk_rating = merchants.risk_rating.str.title()
merchants.risk_rating.unique()


array(['Low', 'Medium', 'High', 'Critical', 'High Risk', nan, 'Unknown'],
      dtype=object)

In [294]:
# replacing null values with"Nan"
import numpy as np
merchants = merchants.replace(np.nan, "Nan")

In [297]:
merchants.isnull().sum()

merchant_id          0
merchant_name        0
merchant_category    0
province             0
city                 0
risk_rating          0
dtype: int64

In [289]:
# Showing the entire row duplicates
merchants.duplicated().sum()

# since the merchant id are unique
merchants['merchant_id'].duplicated().sum()

#viewing duplicates
merchants[merchants['merchant_id'].duplicated(keep=False)]

,merchant_id,merchant_name,merchant_category,province,city,risk_rating


In [290]:
# Removing the duplicates
merchants = merchants.drop_duplicates()

merchants[merchants.duplicated() == True]


,merchant_id,merchant_name,merchant_category,province,city,risk_rating


In [292]:
merchants.province = merchants.province.str.replace(r'\s+', ' ', regex=True)
merchants.province.unique()

array(['Free State', 'Mpumalanga', 'Limpopo', 'Kwazulu-Natal', 'Gauteng',
       'Eastern Cape', 'North West', 'Western Cape', 'Northern Cape'],
      dtype=object)

In [293]:
print(merchants.info())
print(merchants.isnull().sum())
print(merchants.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   merchant_id        20000 non-null  int64 
 1   merchant_name      20000 non-null  object
 2   merchant_category  20000 non-null  object
 3   province           20000 non-null  object
 4   city               20000 non-null  object
 5   risk_rating        20000 non-null  object
dtypes: int64(1), object(5)
memory usage: 937.6+ KB
None
merchant_id          0
merchant_name        0
merchant_category    0
province             0
city                 0
risk_rating          0
dtype: int64
0
